# §13.9.4 — 층·헤드별 싱크의 확인과 축출 실험

> 딥러닝 교재 · 3부 13장 9절 4항 (🐍)
> 선행: §13.9.1(합=1 제약) · §13.9.2(무행동 저장소) · §13.9.5(시스템 제약)

## 이 노트북이 답하는 질문

1. **싱크는 정말 "일이 없을 때"의 주차장인가?** 같은 헤드를 일이 있는 상황과 없는 상황에서 비교한다.
2. **싱크 질량은 출력에 실제로 기여하는가?** 질량과 값-가중 기여를 분리한다.
3. **싱크를 캐시에서 지우면 무슨 일이 벌어지는가?** 섭동의 주입과 흡수를 나눠 잰다.

**예상 실행 시간** CPU 약 2분 (`FAST = True`이면 약 60초).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 과제 — 헤드에게 "일이 없는 순간"이 손실을 지는 언어

무작위 토큰 열에 트리거 토큰이 드문드문 박혀 있고, **트리거 다음 토큰은 트리거 직전
토큰의 복사**다. 그 외 위치의 다음 토큰은 무작위다(주변 분포 예측이 최선). 복사
담당 헤드의 처지에서 보면, 트리거 위치에서는 명확한 일이 있고 **나머지 대부분의
위치에서는 가져올 것이 없다** — 그런데 그 위치들도 손실을 지므로, 엉뚱한 값을
실어 오면 주변 분포 예측이 오염된다. §13.9.1의 무행동 압력이 정확히 이 지점에
걸리고, 소프트맥스는 무행동을 금지하므로 학습은 배수구를 찾아야 한다.

모델은 공용 미니 트랜스포머(2층, $h=4$, $d=48$). 첫 토큰은 BOS로 고정된다.

In [ ]:
# ── 공용 미니 트랜스포머 (NumPy, 완전한 순전파+역전파) ──────────
# 구조: 임베딩 → L × [Pre-LN 블록 (MHA + MLP, 잔차)] → LN → 판독
# 위치 부호화: 'learned' | 'sin' | 'rope' | 'alibi' | 'none'

def make_config(V, d=48, L=2, h=4, T_max=64, pe='learned', causal=True, seed=0):
    dh = d // h
    rn = np.random.default_rng(seed)
    p = {}
    p['emb'] = rn.standard_normal((V, d)) * 0.5 / np.sqrt(d)
    if pe == 'learned':
        p['pos'] = rn.standard_normal((T_max, d)) * 0.5 / np.sqrt(d)
    for l in range(L):
        s = f'l{l}_'
        for nm in ['wq', 'wk', 'wv', 'wo']:
            p[s + nm] = rn.standard_normal((d, d)) / np.sqrt(d)
        p[s + 'ln1g'] = np.ones(d); p[s + 'ln1b'] = np.zeros(d)
        p[s + 'w1'] = rn.standard_normal((d, 4 * d)) / np.sqrt(d)
        p[s + 'b1'] = np.zeros(4 * d)
        p[s + 'w2'] = rn.standard_normal((4 * d, d)) / np.sqrt(4 * d)
        p[s + 'b2'] = np.zeros(d)
        p[s + 'ln2g'] = np.ones(d); p[s + 'ln2b'] = np.zeros(d)
    p['lnfg'] = np.ones(d); p['lnfb'] = np.zeros(d)
    p['out'] = rn.standard_normal((d, V)) / np.sqrt(d)
    cfg = dict(V=V, d=d, L=L, h=h, dh=dh, T_max=T_max, pe=pe, causal=causal)
    return p, cfg

def _sin_pe(T, d):
    pos = np.arange(T)[:, None]
    l2 = np.arange(0, d, 2)[None, :]
    ang = pos / (10000.0 ** (l2 / d))
    pe = np.zeros((T, d))
    pe[:, 0::2] = np.sin(ang); pe[:, 1::2] = np.cos(ang)
    return pe

def _rope_angles(T, dh, scale=1.0):
    pos = np.arange(T)[:, None] * scale
    l2 = np.arange(0, dh, 2)[None, :]
    return pos / (10000.0 ** (l2 / dh))          # (T, dh/2)

def _rope_apply(x, ang, inverse=False):
    # x: (B,h,T,dh) — 짝수/홀수 쌍을 각도 ang(T,dh/2)만큼 회전
    c, s = np.cos(ang), np.sin(ang)
    if inverse:
        s = -s
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return np.stack([x1 * c - x2 * s, x1 * s + x2 * c], axis=-1).reshape(x.shape)

def _alibi_slopes(h):
    return np.array([2.0 ** (-8.0 * (i + 1) / h) for i in range(h)])

def _ln_f(x, g, b):
    mu = x.mean(-1, keepdims=True)
    xc = x - mu
    var = (xc ** 2).mean(-1, keepdims=True)
    inv = 1.0 / np.sqrt(var + 1e-5)
    xh = xc * inv
    return xh * g + b, (xh, inv)

def _ln_b(dy, cache, g):
    xh, inv = cache
    dxh = dy * g
    dg = (dy * xh).sum(axis=tuple(range(dy.ndim - 1)))
    db = dy.sum(axis=tuple(range(dy.ndim - 1)))
    dx = inv * (dxh - dxh.mean(-1, keepdims=True) - xh * (dxh * xh).mean(-1, keepdims=True))
    return dx, dg, db

def forward(p, cfg, idx, targets=None, rope_scale=1.0, head_mask=None,
            skip=None, want_attn=False, kv_keep=None):
    """idx:(B,T) 정수. targets:(B,T) 또는 None.
    head_mask:(L,h) 0/1, skip: {'attn':set(l), 'mlp':set(l)},
    kv_keep:(T,) bool — 열 s의 키·값 사용 여부(캐시 축출 흉내)."""
    B, T = idx.shape
    d, L, h, dh = cfg['d'], cfg['L'], cfg['h'], cfg['dh']
    x = p['emb'][idx]                              # (B,T,d)
    if cfg['pe'] == 'learned':
        x = x + p['pos'][:T]
    elif cfg['pe'] == 'sin':
        x = x + _sin_pe(T, d)
    cache = {'idx': idx, 'T': T, 'B': B, 'xs': [], 'attn': []}
    ang = _rope_angles(T, dh, rope_scale) if cfg['pe'] == 'rope' else None
    if cfg['causal']:
        nmask = np.triu(np.full((T, T), -np.inf), k=1)
    else:
        nmask = np.zeros((T, T))
    if kv_keep is not None:
        nmask = nmask.copy()
        nmask[:, ~kv_keep] = -np.inf
        np.fill_diagonal(nmask, 0.0)      # 자기 자신은 항상 보인다 (캐시 축출 의미론)
    if cfg['pe'] == 'alibi':
        sl = _alibi_slopes(h)
        dist = np.maximum(np.arange(T)[:, None] - np.arange(T)[None, :], 0)
        abias = -sl[:, None, None] * dist[None]    # (h,T,T)
    else:
        abias = np.zeros((1, T, T))
    skip = skip or {'attn': set(), 'mlp': set()}
    attns = []
    def split(z):
        return z.reshape(B, T, h, dh).transpose(0, 2, 1, 3)       # (B,h,T,dh)
    for l in range(L):
        s = f'l{l}_'
        c = {}
        if l not in skip['attn']:
            # ── MHA 가지 ──
            h1, c['ln1'] = _ln_f(x, p[s + 'ln1g'], p[s + 'ln1b'])
            c['h1'] = h1
            q = split(h1 @ p[s + 'wq']); k = split(h1 @ p[s + 'wk']); v = split(h1 @ p[s + 'wv'])
            if cfg['pe'] == 'rope':
                q = _rope_apply(q, ang); k = _rope_apply(k, ang)
            e = np.einsum('bhtd,bhsd->bhts', q, k) / np.sqrt(dh) + nmask + abias[None]
            e -= e.max(-1, keepdims=True)
            a = np.exp(e); a /= a.sum(-1, keepdims=True)
            if head_mask is not None:
                hm = head_mask[l][None, :, None, None]
            else:
                hm = 1.0
            av = np.einsum('bhts,bhsd->bhtd', a, v) * hm
            avm = av.transpose(0, 2, 1, 3).reshape(B, T, d)
            x = x + avm @ p[s + 'wo']
            c.update(q=q, k=k, v=v, a=a, avm=avm, hm=hm)
            attns.append(a)
        else:
            attns.append(None)
        if l not in skip['mlp']:
            # ── MLP 가지 ──
            h2, c['ln2'] = _ln_f(x, p[s + 'ln2g'], p[s + 'ln2b'])
            z1 = h2 @ p[s + 'w1'] + p[s + 'b1']
            r = np.maximum(z1, 0.0)               # ReLU (역전파 단순화)
            x = x + r @ p[s + 'w2'] + p[s + 'b2']
            c['mlp'] = (h2, z1, r)
        else:
            c['mlp'] = None
        cache[s] = c
    hf, cache['lnf'] = _ln_f(x, p['lnfg'], p['lnfb'])
    cache['hf'] = hf
    logits = hf @ p['out']
    cache['logits'] = logits
    out = {'logits': logits}
    if want_attn:
        out['attn'] = attns
    if targets is not None:
        valid = targets >= 0                      # -1 = 손실에서 제외
        tsafe = np.maximum(targets, 0)
        z = logits - logits.max(-1, keepdims=True)
        lse = np.log(np.exp(z).sum(-1))
        ll = z[np.arange(B)[:, None], np.arange(T)[None, :], tsafe] - lse
        out['loss'] = -(ll * valid).sum() / max(valid.sum(), 1)
        P = np.exp(z); P /= P.sum(-1, keepdims=True)
        cache['P'] = P; cache['targets'] = tsafe; cache['valid'] = valid
    out['cache'] = cache
    return out

def backward(p, cfg, cache, rope_scale=1.0):
    B, T = cache['B'], cache['T']
    d, L, h, dh = cfg['d'], cfg['L'], cfg['h'], cfg['dh']
    g = {k: np.zeros_like(v) for k, v in p.items()}
    P, targets, valid = cache['P'], cache['targets'], cache['valid']
    dlogits = P.copy()
    dlogits[np.arange(B)[:, None], np.arange(T)[None, :], targets] -= 1.0
    dlogits *= valid[:, :, None]
    dlogits /= max(valid.sum(), 1)
    hf = cache['hf']
    g['out'] = np.einsum('btd,btv->dv', hf, dlogits)
    dhf = dlogits @ p['out'].T
    dx, g['lnfg'], g['lnfb'] = _ln_b(dhf, cache['lnf'], p['lnfg'])
    ang = _rope_angles(T, dh, rope_scale) if cfg['pe'] == 'rope' else None
    for l in range(L - 1, -1, -1):
        s = f'l{l}_'
        c = cache[s]
        if c['mlp'] is not None:
            h2, z1, r = c['mlp']
            dmlp = dx                                   # 잔차: 가지로 흘러드는 기울기
            g[s + 'w2'] += np.einsum('btf,btd->fd', r, dmlp)
            g[s + 'b2'] += dmlp.sum((0, 1))
            dr = dmlp @ p[s + 'w2'].T
            dz1 = dr * (z1 > 0)
            g[s + 'w1'] += np.einsum('btd,btf->df', h2, dz1)
            g[s + 'b1'] += dz1.sum((0, 1))
            dh2 = dz1 @ p[s + 'w1'].T
            dxi, dg2, db2 = _ln_b(dh2, c['ln2'], p[s + 'ln2g'])
            g[s + 'ln2g'] += dg2; g[s + 'ln2b'] += db2
            dx = dx + dxi
        if 'a' not in c:
            continue
        # MHA 가지
        dattn_out = dx
        g[s + 'wo'] += np.einsum('btd,bte->de', c['avm'], dattn_out)
        davm = dattn_out @ p[s + 'wo'].T
        dav = davm.reshape(B, T, h, dh).transpose(0, 2, 1, 3) * c['hm']
        a, q, k, v = c['a'], c['q'], c['k'], c['v']
        da = np.einsum('bhtd,bhsd->bhts', dav, v)
        dv = np.einsum('bhts,bhtd->bhsd', a, dav)
        de = a * (da - (a * da).sum(-1, keepdims=True))
        dq = np.einsum('bhts,bhsd->bhtd', de, k) / np.sqrt(dh)
        dk = np.einsum('bhts,bhtd->bhsd', de, q) / np.sqrt(dh)
        if cfg['pe'] == 'rope':
            dq = _rope_apply(dq, ang, inverse=True)
            dk = _rope_apply(dk, ang, inverse=True)
        def merge(z):
            return z.transpose(0, 2, 1, 3).reshape(B, T, d)
        dq, dk, dv = merge(dq), merge(dk), merge(dv)
        h1 = c['h1']
        g[s + 'wq'] += np.einsum('btd,bte->de', h1, dq)
        g[s + 'wk'] += np.einsum('btd,bte->de', h1, dk)
        g[s + 'wv'] += np.einsum('btd,bte->de', h1, dv)
        dh1 = dq @ p[s + 'wq'].T + dk @ p[s + 'wk'].T + dv @ p[s + 'wv'].T
        dxi, dg1, db1 = _ln_b(dh1, c['ln1'], p[s + 'ln1g'])
        g[s + 'ln1g'] += dg1; g[s + 'ln1b'] += db1
        dx = dx + dxi
    if cfg['pe'] == 'learned':
        g['pos'][:T] += dx.sum(0)
    np.add.at(g['emb'], cache['idx'], dx)
    return g

def adam_init(p):
    return {k: np.zeros_like(v) for k, v in p.items()}, {k: np.zeros_like(v) for k, v in p.items()}

def adam_step(p, g, m, v, t, lr=3e-3):
    for k in p:
        m[k] = 0.9 * m[k] + 0.1 * g[k]
        v[k] = 0.999 * v[k] + 0.001 * g[k] ** 2
        p[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)

def train_lm(p, cfg, sample_batch, steps, lr=3e-3, log_every=0, rope_scale=1.0):
    m, v = adam_init(p)
    hist = []
    for t in range(1, steps + 1):
        idx, tgt = sample_batch()
        out = forward(p, cfg, idx, targets=tgt, rope_scale=rope_scale)
        g = backward(p, cfg, out['cache'], rope_scale=rope_scale)
        adam_step(p, g, m, v, t, lr)
        hist.append(out['loss'])
        if log_every and t % log_every == 0:
            print(f"  step {t}: loss {np.mean(hist[-log_every:]):.3f}")
    return hist

In [ ]:
V = 32; T_SEQ = 48; TRG = 2

def trig_batch(B, rn):
    idx = rn.integers(3, V, (B, T_SEQ))
    idx[:, 0] = 1                                     # BOS
    for b in range(B):
        qs = rn.choice(np.arange(3, T_SEQ - 1, 3), size=5, replace=False)
        for q in qs:
            idx[b, q] = TRG
            idx[b, q + 1] = idx[b, q - 1]             # 트리거 뒤 = 트리거 앞의 복사
    tgt = np.full((B, T_SEQ), -1)
    tgt[:, :-1] = idx[:, 1:]
    return idx, tgt

p, cfg = make_config(V=V, d=48, L=2, h=4, T_max=T_SEQ, pe='learned', seed=5)
rn = np.random.default_rng(300)
STEPS = 250 if FAST else 500
train_lm(p, cfg, lambda: trig_batch(32, rn), steps=STEPS, lr=3e-3)
idx_ev, tgt_ev = trig_batch(128, np.random.default_rng(41))
out = forward(p, cfg, idx_ev, targets=tgt_ev, want_attn=True)
base = out['loss']
print(f"손실 {base:.3f} (무작위 위치의 하한 근사 = 균등 {np.log(V-3):.2f}) ")

---
## 2. 층·헤드별 싱크 — 일이 있을 때와 없을 때

In [ ]:
is_trg = (idx_ev == TRG)
non = (~is_trg) & (np.arange(T_SEQ)[None, :] >= 6)
mass = np.zeros((2, 4, 2))                            # (층, 헤드, [트리거, 비트리거])
for l in range(2):
    a = out['attn'][l]
    for hh in range(4):
        m0 = a[:, hh, :, 0]
        mass[l, hh, 0] = m0[is_trg].mean()
        mass[l, hh, 1] = m0[non].mean()
    print(f"L{l}: BOS 질량 @트리거(일 있음) {np.round(mass[l,:,0],2)} | @비트리거(일 없음) {np.round(mass[l,:,1],2)}")
l_star, h_star = np.unravel_index(mass[:, :, 1].argmax(), (2, 4))
print(f"→ 싱크 헤드: L{l_star}-H{h_star} (비트리거에서 질량 {mass[l_star,h_star,1]:.2f}, 트리거에서 {mass[l_star,h_star,0]:.2f})")

---
## 3. 질량 대 기여, 그리고 로짓 이상치

In [ ]:
a_star = out['attn'][l_star][:, h_star]               # (B,T,T)
c_star = out['cache'][f'l{l_star}_']
v_star = c_star['v'][:, h_star]                       # (B,T,dh)
alpha0 = a_star[:, :, 0]
# 헤드 출력을 BOS 성분과 내용 성분으로 분해한다
o_full = np.einsum('bts,bsd->btd', a_star, v_star)
o_bos = alpha0[:, :, None] * v_star[:, 0:1, :]        # α₀·v₀ — 고정 방향 위의 스칼라
o_cont = o_full - o_bos
# 각 성분이 몇 차원의 정보를 나르는가 — 주성분 1개의 설명 분산
def evr1(X):
    Xc = X - X.mean(0)
    s = np.linalg.svd(Xc, compute_uv=False)
    return (s[0] ** 2) / (s ** 2).sum()
sel_bt = np.where(non)
evr_bos = evr1(o_bos[sel_bt][::5])
evr_cont = evr1(o_cont[sel_bt][::5])
# 싱크 헤드의 로짓 (스케일 내적) 재구성
q_s, k_s = c_star['q'][:, h_star], c_star['k'][:, h_star]
e_star = np.einsum('btd,bsd->bts', q_s, k_s) / np.sqrt(cfg['dh'])
e0, er_list = [], []
for b in range(0, 128, 4):
    for t in range(6, T_SEQ, 3):
        e0.append(e_star[b, t, 0])                    # BOS로 향하는 로짓
        er_list.append(e_star[b, t, 1:t + 1])         # 인과적으로 보이는 일반 키들
e0 = np.array(e0); er = np.concatenate(er_list)
print(f"싱크 헤드 로짓:  BOS행 평균 {e0.mean():+.1f}  |  일반 키 평균 {er.mean():+.1f}")
print(f"헤드 출력 성분의 1주성분 설명 분산: BOS 성분 {evr_bos:.3f}  |  내용 성분 {evr_cont:.3f}")
print("→ 주차 성분은 고정 방향 위의 스칼라 하나(부기 신호)로 완전히 요약된다 — 내용을 나르지 않는다")

---
## 4. 축출 — 섭동의 주입과 흡수

In [ ]:
kv0 = np.ones(T_SEQ, bool); kv0[0] = False            # 싱크(BOS) 축출
kvr = np.ones(T_SEQ, bool); kvr[11] = False           # 무작위 내용 토큰 축출
out0 = forward(p, cfg, idx_ev, targets=tgt_ev, kv_keep=kv0)
outr = forward(p, cfg, idx_ev, targets=tgt_ev, kv_keep=kvr)
dlogit0 = np.abs(out0['logits'] - out['logits']).mean()
dlogitr = np.abs(outr['logits'] - out['logits']).mean()
print(f"손실:  원본 {base:.3f} | BOS 축출 {out0['loss']:.3f} | 내용 축출 {outr['loss']:.3f}")
print(f"로짓 섭동(평균 |Δ|):  BOS 축출 {dlogit0:.3f} | 내용 축출 {dlogitr:.3f}")

---
## 5. 교재 그림 — fig_13_9_4

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 층·헤드별 BOS 질량 — 상황별
ax = axes[0]
xs = np.arange(8)
ax.bar(xs - 0.19, mass[:, :, 0].ravel(), 0.38, color=CB[3], label=lab('일이 있을 때 (트리거)', 'has job'))
ax.bar(xs + 0.19, mass[:, :, 1].ravel(), 0.38, color=CB[4], label=lab('일이 없을 때 (비트리거)', 'no job'))
ax.set_xticks(xs)
ax.set_xticklabels([f'L{l}H{h}' for l in range(2) for h in range(4)], fontsize=8)
ax.set_ylabel(lab('BOS(첫 토큰) 어텐션 질량', 'mass on BOS'))
ax.set_title(lab('(a) 같은 헤드가 일이 없을 때만 주차한다', '(a) sink by situation'), fontsize=10)
ax.legend(fontsize=8)

# (b) 헤드 출력의 분해 — 주차 성분은 정보를 나르지 않는다
ax = axes[1]
xs_b = np.arange(2)
ax.bar(xs_b, [alpha0[non].mean(), 1 - alpha0[non].mean()], 0.5,
       color=[CB[4], CB[2]], alpha=0.9)
ax.set_xticks(xs_b)
ax.set_xticklabels([lab('BOS 성분', 'BOS part'), lab('내용 성분', 'content part')], fontsize=9)
ax.set_ylabel(lab('어텐션 질량 (비트리거 평균)', 'attn mass'), color='k')
ax2b = ax.twinx()
ax2b.plot(xs_b, [evr_bos, evr_cont], 'D--', color=CB[5], ms=7, lw=1)
ax2b.set_ylim(0, 1.1)
ax2b.set_ylabel(lab('1주성분 설명 분산 (1.0 = 방향 하나뿐)', 'top-1 PC EVR'), color=CB[5])
ax2b.grid(False)
ax.set_title(lab(f'(b) L{l_star}H{h_star}: 주차 성분은 방향 하나에 갇혀 있다', '(b) parked component is rank-1'), fontsize=10)

# (c) 로짓 이상치
ax = axes[2]
bins = np.linspace(min(er.min(), e0.min()), max(er.max(), e0.max()), 50)
ax.hist(er, bins=bins, color=CB[2], alpha=0.8, density=True, label=lab('일반 키로의 로짓', 'regular keys'))
ax.hist(e0, bins=bins, color=CB[4], alpha=0.8, density=True, label=lab('BOS로의 로짓', 'to BOS'))
ax.set_xlabel(lab('어텐션 로짓', 'attention logit'))
ax.set_ylabel(lab('밀도', 'density'))
ax.set_title(lab('(c) 싱크는 로짓 분포의 이상치 무리를 만든다', '(c) logit outliers'), fontsize=10)
ax.legend(fontsize=8)

# (d) 축출 — 손실과 섭동
ax = axes[3]
xs = np.arange(2); w = 0.38
ax.bar(xs - w/2, [out0['loss'] - base, outr['loss'] - base], w, color=CB[5],
       label=lab('Δ손실', 'Δloss'))
ax.set_xticks(xs)
ax.set_xticklabels([lab('BOS(싱크) 축출', 'evict BOS'), lab('내용 토큰 축출', 'evict content')], fontsize=9)
ax.set_ylabel(lab('손실 증가', '$\\Delta$ loss'), color=CB[5])
ax2 = ax.twinx()
ax2.bar(xs + w/2, [dlogit0, dlogitr], w, color=CB[1])
ax2.set_ylabel(lab('로짓 섭동 평균 $|\\Delta|$', 'logit perturbation'), color=CB[1])
ax2.grid(False)
ax.set_title(lab('(d) 섭동은 주입되지만, 이 규모에선 흡수된다', '(d) eviction'), fontsize=10)
ax.legend(fontsize=8, loc='upper left')

save_book_fig(fig, 'fig_13_9_4')
plt.show()

> ### 읽는 법
>
> (a) 이 그림이 싱크의 기능적 정의다. 싱크 헤드는 트리거 위치(일이 있음)에서는 BOS에
> 질량을 거의 주지 않다가, 비트리거 위치(가져올 것이 없음)에서는 절반 가까이를 BOS에
> 주차한다. **싱크는 위치의 속성이 아니라 "일 없는 헤드"의 행동이다**(§13.9.2).
> (b) 왜 무해한 주차인가 — 주차 성분 $\alpha_0 v_0$는 고정 방향 $v_0$ 위의 스칼라라서
> 1주성분이 분산의 100%를 설명한다(정확히 계수 1). 나르는 것은 내용이 아니라 "지금
> 얼마나 주차 중인가"라는 **부기\mycmt{bookkeeping} 신호 하나**뿐이고, 내용 성분은
> 여러 방향에 정보를 편다. 어텐션 질량 지도는 이 구분을 보여 주지 못한다(§13.9.6,
> §13.11의 예고). (c) 그 주차는 로짓 공간에서 **분리된 이상치 무리**로 구현된다 —
> §13.9.5에서 말한 양자화의 골칫거리가 바로 이 무리다.
> (d) 정직한 규모 주석. BOS를 축출하면 로짓 섭동이 내용 토큰 축출보다 크게 주입되지만,
> 이 작은 모델은 그 오염을 흡수해 손실이 거의 움직이지 않는다. 수십 층 × 수백 헤드가
> 같은 배수구를 공유하는 실규모 모델에서 같은 축출이 생성을 무너뜨린다는 것 — 그리고
> 최근 창에 첫 토큰 몇 개만 남겨도 복구된다는 것 — 이 스트리밍 추론 연구의 관찰이다
> (Xiao et al. 2024). 씨앗은 이 규모에도 있고, 발아는 규모의 문제다.

---
## 6. 자기 점검

1. (a)에서 싱크가 L0에 나타났다면, L1이 아니라 L0인 이유를 생각해 보라. 복사 헤드는 어느 층에 있는가?
2. BOS 대신 임의 위치의 토큰을 "항상 보이는 전용 싱크"로 학습에 추가하면(§13.9.3의 레지스터 토큰) (a)의 주차장이 옮겨 가는가? 실험해 보라.
3. (c)의 이상치 무리를 없애는 §13.9.3의 세 처방 중 소급 적용(재학습 없이)이 가능한 것은 무엇인가?
4. softmax 분모에 1을 더한 버전(softmax$_1$)으로 같은 모델을 다시 학습하면 (a)의 주차 질량이 어디로 가는가?

## 7. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| 트리거 수 | 1절 | 5 | "일 없는 순간"의 비율 |
| `V` | 1절 | 32 | 주변 분포 예측의 난도 |
| `L`, `h` | 1절 | 2, 4 | 배수구를 공유하는 헤드 수 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")